# S6.7 — Agent Orchestrator (LangGraph)

Verifies that the `AgenticRAGService` compiles a LangGraph `StateGraph` wiring together
guardrail, retrieval, grading, rewrite, and generation nodes into a complete agentic RAG workflow.

In [1]:
import sys
sys.path.insert(0, "../..")

from src.services.agents.agentic_rag import AgenticRAGService, AgenticRAGResponse, ainvoke_out_of_scope_step
from src.services.agents.factory import create_agentic_rag_service
from src.services.agents.state import create_initial_state
from src.services.agents.models import SourceItem, GuardrailScoring, GradingResult
from src.services.rag.models import SourceReference

print("All imports successful")

All imports successful


## 1. AgenticRAGResponse Model

In [2]:
# Test response model with defaults and explicit fields
resp = AgenticRAGResponse(answer="Test answer [1].")
assert resp.sources == []
assert resp.reasoning_steps == []
assert resp.metadata == {}

resp2 = AgenticRAGResponse(
    answer="Transformers [1].",
    sources=[SourceReference(index=1, arxiv_id="1706.03762", title="Attention", authors=["Vaswani"], arxiv_url="https://arxiv.org/abs/1706.03762")],
    reasoning_steps=["Guardrail: passed", "Retrieved 5 docs"],
    metadata={"elapsed_seconds": 1.5},
)
assert len(resp2.sources) == 1
assert resp2.sources[0].arxiv_id == "1706.03762"
print(f"Response: {resp2.answer}")
print(f"Sources: {len(resp2.sources)}")
print(f"Steps: {resp2.reasoning_steps}")
print("\n✓ AgenticRAGResponse model works correctly")

Response: Transformers [1].
Sources: 1
Steps: ['Guardrail: passed', 'Retrieved 5 docs']

✓ AgenticRAGResponse model works correctly


## 2. Graph Compilation

In [3]:
from unittest.mock import MagicMock

mock_llm = MagicMock()
mock_pipeline = MagicMock()

service = AgenticRAGService(llm_provider=mock_llm, retrieval_pipeline=mock_pipeline)
assert service._compiled_graph is not None

graph = service._compiled_graph.get_graph()
node_ids = set(graph.nodes)
expected = {"guardrail", "out_of_scope", "retrieve", "grade_documents", "rewrite_query", "generate_answer"}
assert expected.issubset(node_ids), f"Missing nodes: {expected - node_ids}"

print(f"Graph nodes: {sorted(node_ids)}")
print(f"All expected nodes present: {expected}")
print("\n✓ Graph compiles successfully with all 6 nodes")

Graph nodes: ['__end__', '__start__', 'generate_answer', 'grade_documents', 'guardrail', 'out_of_scope', 'retrieve', 'rewrite_query']
All expected nodes present: {'out_of_scope', 'rewrite_query', 'retrieve', 'grade_documents', 'guardrail', 'generate_answer'}

✓ Graph compiles successfully with all 6 nodes


## 3. Out-of-Scope Node

In [4]:
from langchain_core.messages import AIMessage

state = create_initial_state("How to make pizza?")
result = ainvoke_out_of_scope_step(state)

assert "messages" in result
msg = result["messages"][0]
assert isinstance(msg, AIMessage)
assert "research assistant" in msg.content.lower()
print(f"Out-of-scope response: {msg.content[:100]}...")
print("\n✓ Out-of-scope node returns polite rejection")

Out-of-scope response: I'm a research assistant focused on academic papers. I can't help with that topic, but I'd be happy ...

✓ Out-of-scope node returns polite rejection


## 4. Result Extraction Helpers

In [5]:
from langchain_core.messages import HumanMessage

# _extract_answer
state_with_ai = {"messages": [HumanMessage(content="Q?"), AIMessage(content="Answer [1].")]}
assert AgenticRAGService._extract_answer(state_with_ai) == "Answer [1]."
assert "unable" in AgenticRAGService._extract_answer({"messages": []}).lower() or len(AgenticRAGService._extract_answer({"messages": []})) > 0
print("✓ _extract_answer works")

# _extract_sources
state_sources = {
    "relevant_sources": [
        SourceItem(arxiv_id="1706.03762", title="Attention", authors=["Vaswani"], url="https://arxiv.org/abs/1706.03762", chunk_text="text"),
        SourceItem(arxiv_id="1810.04805", title="BERT", authors=["Devlin"], url="https://arxiv.org/abs/1810.04805", chunk_text="text2"),
    ]
}
sources = AgenticRAGService._extract_sources(state_sources)
assert len(sources) == 2
assert sources[0].index == 1
assert sources[1].index == 2
assert sources[0].arxiv_id == "1706.03762"
assert AgenticRAGService._extract_sources({}) == []
print("✓ _extract_sources works")

# _extract_reasoning_steps
state_full = {
    "guardrail_result": GuardrailScoring(score=85, reason="Research"),
    "sources": [SourceItem(arxiv_id="1", title="P", authors=[], url="u", chunk_text="c")],
    "retrieval_attempts": 1,
    "grading_results": [GradingResult(document_id="1", is_relevant=True, score=1.0)],
    "relevant_sources": [SourceItem(arxiv_id="1", title="P", authors=[], url="u", chunk_text="c")],
    "messages": [AIMessage(content="Answer")],
    "metadata": {},
}
steps = AgenticRAGService._extract_reasoning_steps(state_full)
assert len(steps) >= 3
combined = " ".join(steps).lower()
assert "guardrail" in combined
assert "retriev" in combined
print(f"Reasoning steps: {steps}")
print("\n✓ All extraction helpers work correctly")

✓ _extract_answer works
✓ _extract_sources works
Reasoning steps: ['Guardrail: passed (score=85, reason=Research)', 'Retrieved 1 documents (attempt 1)', 'Grading: 1/1 relevant', 'Generated answer with 1 citations']

✓ All extraction helpers work correctly


## 5. Factory Function

In [6]:
# Factory creates service
svc = create_agentic_rag_service(llm_provider=MagicMock(), retrieval_pipeline=MagicMock())
assert isinstance(svc, AgenticRAGService)
print("✓ Factory creates AgenticRAGService")

# Factory rejects None llm_provider
try:
    create_agentic_rag_service(llm_provider=None)
    assert False, "Should have raised ValueError"
except ValueError as e:
    print(f"✓ Factory rejects None llm_provider: {e}")

print("\n✓ Factory function works correctly")

✓ Factory creates AgenticRAGService
✓ Factory rejects None llm_provider: llm_provider is required and cannot be None

✓ Factory function works correctly


## 6. End-to-End: Happy Path with Mocked LLM

In [7]:
import asyncio
from unittest.mock import AsyncMock, MagicMock
from src.services.agents.models import GradeDocuments
from src.schemas.api.search import SearchHit

# Build mocks
mock_llm = MagicMock()
mock_pipeline = AsyncMock()

# Retrieval returns a paper
mock_pipeline.retrieve = AsyncMock(return_value=MagicMock(
    results=[SearchHit(
        arxiv_id="1706.03762", title="Attention Is All You Need",
        authors=["Vaswani et al."], score=0.95,
        chunk_text="The dominant sequence transduction models are based on complex recurrent or convolutional neural networks.",
        pdf_url="",
    )],
    stages_executed=["hybrid_search", "rerank"],
    total_candidates=20, timings={"hybrid_search": 0.1},
))

# LLM responses: guardrail pass, grade yes, then generate
guardrail_resp = GuardrailScoring(score=85, reason="Research about transformers")
grade_resp = GradeDocuments(binary_score="yes", reasoning="Directly relevant")
call_count = 0

async def structured_side_effect(prompt):
    global call_count
    call_count += 1
    if call_count == 1:
        return guardrail_resp
    return grade_resp

structured = AsyncMock()
structured.ainvoke = structured_side_effect

model = MagicMock()
model.with_structured_output.return_value = structured
model.ainvoke = AsyncMock(return_value=AIMessage(
    content="Transformers are based on self-attention mechanisms [1].\n\n**Sources:**\n1. [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — Vaswani et al."
))
mock_llm.get_langchain_model.return_value = model

# Run
service = AgenticRAGService(llm_provider=mock_llm, retrieval_pipeline=mock_pipeline)
result = await service.ask("What are transformers?")

print(f"Answer: {result.answer[:120]}...")
print(f"Sources: {len(result.sources)}")
print(f"Reasoning: {result.reasoning_steps}")
print(f"Elapsed: {result.metadata.get('elapsed_seconds', '?')}s")
assert len(result.answer) > 0
assert isinstance(result.metadata, dict)
print("\n✓ End-to-end happy path works correctly")

Answer: Transformers are based on self-attention mechanisms [1].

**Sources:**
1. [Attention Is All You Need](https://arxiv.org/...
Sources: 1
Reasoning: ['Guardrail: passed (score=85, reason=Research about transformers)', 'Retrieved 1 documents (attempt 1)', 'Grading: 1/1 relevant', 'Generated answer with 1 citations']
Elapsed: 0.007s

✓ End-to-end happy path works correctly
